In [2]:
import pandas as pd
from pathlib import Path
import yaml

def find_project_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / "pyproject.toml").exists() or (parent / ".git").exists():
            return parent
    raise RuntimeError("Project root not found (pyproject.toml/.git)")

PROJECT_ROOT = find_project_root()
CONFIG_ROOT = PROJECT_ROOT / "config"
dataset_cfg_path = CONFIG_ROOT / "datasets" / "dataset_config.yml"

with open(dataset_cfg_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

ds = cfg["datasets"]["microsoft_security_incident"]
train_rel = ds["paths"]["train"]
csv_params = ds.get("csv_params", {})

train_path = PROJECT_ROOT / train_rel

df_head = pd.read_csv(
    train_path,
    sep=csv_params.get("sep", ","),
    encoding=csv_params.get("encoding", "utf-8"),
    decimal=csv_params.get("decimal", "."),
    low_memory=csv_params.get("low_memory", False),
    nrows=10
)
df_head

,Id,OrgId,IncidentId,AlertId,Timestamp,DetectorId,AlertTitle,Category,MitreTechniques,IncidentGrade,...,ResourceType,Roles,OSFamily,OSVersion,AntispamDirection,SuspicionLevel,LastVerdict,CountryCode,State,City
0,180388628218,0,612,123247,2024-06-04T06:05:15.000Z,7,6,InitialAccess,NaN,TruePositive,...,NaN,NaN,5,66,NaN,NaN,NaN,31,6,3
1,455266534868,88,326,210035,2024-06-14T03:01:25.000Z,58,43,Exfiltration,NaN,FalsePositive,...,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630
2,1056561957389,809,58352,712507,2024-06-13T04:52:55.000Z,423,298,InitialAccess,T1189,FalsePositive,...,NaN,NaN,5,66,NaN,Suspicious,Suspicious,242,1445,10630
3,1279900258736,92,32992,774301,2024-06-10T16:39:36.000Z,2,2,CommandAndControl,NaN,BenignPositive,...,NaN,NaN,5,66,NaN,Suspicious,Suspicious,242,1445,10630
4,214748368522,148,4359,188041,2024-06-15T01:08:07.000Z,9,74,Execution,NaN,TruePositive,...,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630
5,1322849927433,11,417400,825450,2024-06-10T13:30:56.000Z,0,0,InitialAccess,T1078;T1078.004,FalsePositive,...,NaN,NaN,5,66,NaN,NaN,NaN,8,6,3
6,163208760309,522,566,705663,2024-06-14T23:19:45.000Z,2,2,CommandAndControl,NaN,BenignPositive,...,NaN,NaN,5,66,NaN,Suspicious,Suspicious,242,1445,10630
7,1400159339557,125,38679,47423,2024-06-06T13:39:23.000Z,313,3919,Exfiltration,NaN,BenignPositive,...,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630
8,1219770713645,21,414,197969,2024-06-09T10:21:29.000Z,3,4,SuspiciousActivity,NaN,BenignPositive,...,NaN,NaN,5,66,NaN,Suspicious,Suspicious,242,1445,10630
9,1073741827836,72,70,831157,2024-06-08T02:08:01.000Z,4,3,InitialAccess,NaN,TruePositive,...,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630


In [10]:
import duckdb

ORG_ID = 11
INCIDENT_ID = 417400

query = f"""
SELECT
  COUNT(*) AS n_rows,
  COUNT(DISTINCT Id) AS n_distinct_id,
  COUNT(DISTINCT AlertId) AS n_distinct_alert,
  COUNT(DISTINCT CAST(Timestamp AS VARCHAR)) AS n_distinct_ts
FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')
WHERE OrgId = {ORG_ID} AND IncidentId = {INCIDENT_ID}
"""
duckdb.query(query).to_df()

,n_rows,n_distinct_id,n_distinct_alert,n_distinct_ts
0,4,1,1,1


In [11]:
query = f"""
SELECT *
FROM (
  SELECT
    Id, OrgId, IncidentId, AlertId, Timestamp, DetectorId, Category, MitreTechniques, IncidentGrade, AlertTitle,
    ROW_NUMBER() OVER (
      PARTITION BY OrgId, IncidentId, AlertId, Timestamp, DetectorId, Category
      ORDER BY Id
    ) AS rn
  FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')
  WHERE OrgId = {ORG_ID} AND IncidentId = {INCIDENT_ID}
)
WHERE rn = 1
ORDER BY Timestamp
"""
incident_df = duckdb.query(query).to_df()
incident_df

,Id,OrgId,IncidentId,AlertId,Timestamp,DetectorId,Category,MitreTechniques,IncidentGrade,AlertTitle,rn
0,1322849927433,11,417400,825450,2024-06-10 15:30:56+02:00,0,InitialAccess,T1078;T1078.004,FalsePositive,0,1


In [18]:
query = f"""
SELECT DISTINCT
  *
FROM read_csv_auto('{train_path}', delim='{csv_params.get("sep", ",")}')
WHERE OrgId = {ORG_ID} AND IncidentId = {INCIDENT_ID}
ORDER BY Timestamp
"""
incident_df = duckdb.query(query).to_df()

import pandas as pd
pd.set_option("display.max_columns", None)

incident_df
#incident_df.iloc[0].to_frame("value")

,Id,OrgId,IncidentId,AlertId,Timestamp,DetectorId,AlertTitle,Category,MitreTechniques,IncidentGrade,ActionGrouped,ActionGranular,EntityType,EvidenceRole,DeviceId,Sha256,IpAddress,Url,AccountSid,AccountUpn,AccountObjectId,AccountName,DeviceName,NetworkMessageId,EmailClusterId,RegistryKey,RegistryValueName,RegistryValueData,ApplicationId,ApplicationName,OAuthApplicationId,ThreatFamily,FileName,FolderPath,ResourceIdName,ResourceType,Roles,OSFamily,OSVersion,AntispamDirection,SuspicionLevel,LastVerdict,CountryCode,State,City
0,1322849927433,11,417400,825450,2024-06-10 15:30:56+02:00,0,0,InitialAccess,T1078;T1078.004,FalsePositive,None,None,CloudLogonSession,Related,98799,138268,360606,160396,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,None,289573,117668,3586,None,None,5,66,None,None,None,242,1445,10630
1,1322849927433,11,417400,825450,2024-06-10 15:30:56+02:00,0,0,InitialAccess,T1078;T1078.004,FalsePositive,None,None,CloudLogonRequest,Related,98799,138268,360606,160396,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,None,289573,117668,3586,None,None,5,66,None,None,None,242,1445,10630
2,1322849927433,11,417400,825450,2024-06-10 15:30:56+02:00,0,0,InitialAccess,T1078;T1078.004,FalsePositive,None,None,User,Impacted,98799,138268,360606,160396,28670,48804,29043,32324,153085,529644,NaN,1631,635,860,2251,3421,881,None,289573,117668,3586,None,None,5,66,None,None,None,242,1445,10630
3,1322849927433,11,417400,825450,2024-06-10 15:30:56+02:00,0,0,InitialAccess,T1078;T1078.004,FalsePositive,None,None,Ip,Related,98799,138268,30410,160396,441377,673934,425863,453297,153085,529644,NaN,1631,635,860,2251,3421,881,None,289573,117668,3586,None,None,5,66,None,None,None,8,6,3


In [ ]:
ORG_ID = 11
INCIDENT_ID = 417400

df_min = pd.read_parquet("GUIDE_train_min.parquet")
incident_df = df_min[(df_min.OrgId==ORG_ID) & (df_min.IncidentId==INCIDENT_ID)].sort_values("Timestamp")
incident_df

In [3]:
from src.crispdm.api.preview_facade_api import run_preview
from pathlib import Path
from crispdm.core.logging_utils_core import init_logging, build_log_file, get_logger

def find_project_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / "pyproject.toml").exists() or (parent / ".git").exists():
            return parent
    raise RuntimeError("Project root not found (pyproject.toml/.git)")

PROJECT_ROOT = find_project_root()
CONFIG_ROOT = PROJECT_ROOT / "config"

# ---------- Paths (IMPORTANT) ----------
preset_path = CONFIG_ROOT / "pipelines" / "classification_pipeline_config.yml"
dataset_cfg_path = CONFIG_ROOT / "datasets" / "dataset_config.yml"
# ---------- Logging ----------
OUT_ROOT = PROJECT_ROOT / "out"




In [4]:
log_file = build_log_file(output_root=OUT_ROOT, run_name="config_smoke_test")
init_logging(log_file, level="DEBUG")
log = get_logger(__name__)
log.info("=== CONFIG SMOKE TEST START ===")
res = run_preview(
    pipeline_config_path = preset_path,
    dataset_config_path = dataset_cfg_path,
    dataset_key="microsoft_security_incident" ,
    notebook_vars={
        "dataset_path": None,
        "target_col": None,
        "time_col": None,
        "id_cols": None,
        "output_root": str(OUT_ROOT),
    }
)
res.suggestions


[10:36:22] [DEBUG] MPPRAI - Logging initialized. log_file=K:\00_Code\DataScience\Project_DS_Microsoft_Security_Incident_Prediction\out\logs\config_smoke_test_20260302_103622.log level=DEBUG
[10:36:22] [INFO] MPPRAI.__main__ - === CONFIG SMOKE TEST START ===
[10:36:22] [INFO] MPPRAI.src.crispdm.api.preview_facade_api - Start [build_factory_config.build_preview_config]...with params: pipeline_config_path=K:\00_Code\DataScience\Project_DS_Microsoft_Security_Incident_Prediction\config\pipelines\classification_pipeline_config.yml dataset_config_path=K:\00_Code\DataScience\Project_DS_Microsoft_Security_Incident_Prediction\config\datasets\dataset_config.yml dataset_key=microsoft_security_incident notebook_vars={'dataset_path': None, 'target_col': None, 'time_col': None, 'id_cols': None, 'output_root': 'K:\\00_Code\\DataScience\\Project_DS_Microsoft_Security_Incident_Prediction\\out'}
[10:36:22] [INFO] MPPRAI.crispdm.config.build_factory_config - Start [build_preview_config] notebooks_vars={'d

{'dataset_path': 'data/raw/train/GUIDE_Train.csv',
 'rows_profiled': 200000,
 'cols': 45,
 'suggestions': {'target_candidates': ['IncidentGrade',
   'Category',
   'EntityType',
   'EvidenceRole',
   'RegistryKey',
   'RegistryValueName',
   'RegistryValueData',
   'ApplicationName',
   'ResourceIdName',
   'OSFamily'],
  'time_candidates': ['Timestamp'],
  'id_candidates': ['Id',
   'OrgId',
   'IncidentId',
   'AlertId',
   'DetectorId',
   'DeviceId',
   'AccountSid',
   'AccountObjectId',
   'NetworkMessageId',
   'EmailClusterId']},
 'artifacts': {'profile_table_png': 'K:/00_Code/DataScience/Project_DS_Microsoft_Security_Incident_Prediction/out/tables_png/stage2/stage2_profile_top25.png'}}